In [ ]:
%cd /content/
!git clone https://github.com/Bekci/ESCAPE.git
%cd /content/ESCAPE

In [ ]:
# Change the id of the Shapenet
# As installing libraries takes a lot of time, you can run downloading and
# unzipping shapenet in terminal to parallelize operations
!gdown <shapenet_dataset_id>

In [ ]:
!unzip -q ShapeNetCompletion.zip

In [ ]:
# This step takes ~20mins
!bash scripts/install.sh

In [ ]:
# Download val_best k16_ns16_thres_0.5_factor_0.5_zero_0.05_rad_0.075
!gdown 1n_DZwe9MDisOK--qCDwzX9aBUcpY-jGR

In [ ]:
%cd /content/ESCAPE

In [ ]:
#Might require if below imports fail
#!pip uninstall datasets --y

In [ ]:
import os
import torch
import numpy as np
from datasets.utils import read_yaml
from datasets.PCN import get_dataset_and_loader
from models.pointr.escape import ESCAPE
from tqdm import tqdm
from utils.training import var_or_cuda, var_keys_for_gpu
from loss_functions.chamfer_ndim import CDNLoss
from utils.optimization import find_points_from_distance

In [ ]:
def optimize_points_loader(model, dataloader, loss_object):
    losses = []
    sup_losses = []
    model.eval()
    results_per_cat = {}

    pbar = tqdm(dataloader)
    with torch.no_grad():
        for (taxonomy_ids, model_ids, data) in pbar:

            for k in data:
                if k in var_keys_for_gpu:
                    data[k] = var_or_cuda(data[k])

            gt_dists = torch.sqrt(torch.sum(1e-6 +  (data['gt'][:,:,None,:] - data['basis_points'][:,None,:,:])**2, axis=-1))

            ret = model(data['partial'], data['basis_points'])

            for i in range(ret[0].shape[0]):

                coarse_points = ret[0][i]
                dense_points = ret[-1][i]
                category_id = taxonomy_ids[i]

                basis_points_np = data['basis_points'].cpu().numpy()[i]
                distances_np  = dense_points.cpu().numpy()
                gt_dists_np = gt_dists.cpu().numpy()[i]

                optimized_points = find_points_from_distance(distances_np, basis_points_np)
                loss = loss_object.chamfer_loss(torch.unsqueeze(torch.from_numpy(optimized_points.astype('float32')), 0).cuda(), data['gt'][i:i+1])
                loss_val = loss.item()
                losses.append(loss_val)

                if category_id not in results_per_cat:
                    results_per_cat[category_id] = []

                results_per_cat[category_id].append(loss_val * 1000)
                pbar.set_description("Loss: %.2f" % (np.mean(losses) * 1000))

    return losses, results_per_cat

In [ ]:
keypoint_number = 8
keypoint_type = 'curvature_radius'
model_config_path = './configs/escape.yaml'
model_save_path = 'val_best.pth'
curvature_params = {}
curvature_params['k'] =  16
curvature_params['curvature_radius'] = 0.075
curvature_params['curvature_thres'] = 0.5

In [ ]:
config_path = './configs/pcn.yaml'
config = read_yaml(config_path)

model_config = read_yaml(model_config_path)

model = ESCAPE(model_config.model).cuda()

if torch.cuda.is_available():
    model = torch.nn.DataParallel(model).cuda()

saved_model_config = torch.load(model_save_path, weights_only=False)

In [ ]:
model.load_state_dict(saved_model_config['model_state_best_val'])

In [ ]:
loss_obj = CDNLoss(mode="point", metric_fun='cd_l1')
sample_num = 5000

In [ ]:
# Random inference from test split
from utils.visualization import plot_points

inference_dataset, inference_loader = get_dataset_and_loader(
    'test',
    config, 1, sample_num, shuffle=False, drop_last=False,
    num_keypoints=keypoint_number, keypoint_type=keypoint_type,
    curvature_params=curvature_params)

rand_idx = np.random.randint(0, len(inference_dataset))
taxonomy_ids, model_ids, data = inference_dataset[rand_idx]

with torch.no_grad():
  basis_points = torch.from_numpy(data['basis_points']).unsqueeze(0)

  ret = model(data['partial'].unsqueeze(0).cuda(), basis_points.cuda())

  dense_points = ret[-1][0]

  basis_points_np = data['basis_points']
  distances_np  = dense_points.cpu().numpy()

  optimized_points = find_points_from_distance(distances_np, basis_points_np)

  plot_points([data['partial'].numpy(), basis_points_np], [2,4])
  plot_points([data['gt'].numpy()], [2])
  plot_points([optimized_points], [2])

  loss = loss_obj.chamfer_loss(torch.from_numpy(optimized_points.astype('float32')).unsqueeze(0).cuda(), data['gt'].unsqueeze(0).cuda() )
  loss_val = loss.item()
  print("CD L1 loss: %.3f " % (loss_val*1000))

In [ ]:
# Inference test split

sample_num = 5000
opt_val_dataset, opt_val_loader = get_dataset_and_loader(
    'test',
    config, 4, sample_num, shuffle=True, drop_last=False,
    num_keypoints=keypoint_number, keypoint_type=keypoint_type,
    curvature_params=curvature_params)

point_losses, opt_res_per_cat = optimize_points_loader(model, opt_val_loader, loss_obj)

mean_loss = np.mean(point_losses)
print("Optimized point loss: %.2f " % (mean_loss*1e3))

for cat in opt_res_per_cat:
    print("Distance loss %.4f for %s" % (np.mean(opt_res_per_cat[cat]), cat))